# BSD vs ONS Business Dynamism Validation

This notebook compares dynamism measures computed from the BSD (Business Structure Database) against the ONS Annual Business Dynamism publication. The aim is to validate that BSD-derived rates are consistent with the official ONS series.

**Measures compared:** Entry rates, exit rates, job reallocation rates (JRR), job creation rates (JCR: entry vs incumbent), job destruction rates (JDR: exit vs incumbent).

**Breakdowns:** Total economy, firm size, firm age, sector.

**Note on methodology:**
- All BSD rates use lagged (t-1) denominators, consistent with ONS methodology.
- For **total economy, size, and age** breakdowns: the denominator is the lagged **total economy** firm count (entry/exit) or employment (job flows), so sub-group rates sum to the total. This matches the ONS presentation in Tables 10 and 13.
- For **sector** breakdowns: the denominator is the lagged **sector-specific** employment/firm count, giving within-sector rates. Note this differs from ONS Table 19, which uses total economy as the denominator — so the BSD sector rates here are not directly additive to the total, but reflect the intensity of reallocation within each sector.
- BSD entry rate = (n_entrants + n_entry_and_exit) / n_firms_{t-1}
- BSD exit rate = (n_exiters + n_entry_and_exit) / n_firms_{t-1}
- JRR = (JC_entrants + JC_incumbents + JD_exiters + JD_incumbents) / emp_{t-1}
- JCR = (JC_entrants + JC_incumbents) / emp_{t-1}
- JDR = (JD_exiters + JD_incumbents) / emp_{t-1}

**Sector mapping:** BSD sectors are mapped to ONS SIC-based industry groups where feasible. Several BSD sectors correspond to sub-sections of ONS industry groups, so sector comparisons are approximate.

In [38]:
import sys
sys.path.append('.')

import pandas as pd
import numpy as np
import altair as alt
from eco_style import pallete
alt.themes.enable('report')

# Suppress pandas warnings
pd.options.mode.chained_assignment = None

BSD_PATH = '../Data/BSD/29_02_2026_BSD_Dynamism_Stats.xlsx'
ONS_PATH = '../Data/ONS/annualbusinessdynamism20012024.xlsx'

## 1. Load and Prepare BSD Data

In [39]:
# --- Load BSD sheets ---
bsd_fd = pd.read_excel(BSD_PATH, sheet_name='firm_dynamics')
bsd_jf = pd.read_excel(BSD_PATH, sheet_name='job_flows')

# Convert disclosure-suppressed '*' cells to NaN (per CLAUDE.md)
raw_count_cols_fd = ['n_firms', 'employment', 'n_entrants', 'n_exiters',
                     'n_entry_and_exit', 'n_reactivations', 'n_incumbents']
raw_count_cols_jf = ['n_firms', 'employment', 'job_creation_entrants',
                     'job_creation_incumbents', 'job_destruction_exiters',
                     'job_destruction_incumbents']
for col in raw_count_cols_fd:
    if col in bsd_fd.columns:
        bsd_fd[col] = pd.to_numeric(bsd_fd[col], errors='coerce')
for col in raw_count_cols_jf:
    if col in bsd_jf.columns:
        bsd_jf[col] = pd.to_numeric(bsd_jf[col], errors='coerce')

# Merge on common keys
bsd = bsd_fd.merge(
    bsd_jf[['year', 'dimension', 'category',
             'job_creation_entrants', 'job_creation_incumbents',
             'job_destruction_exiters', 'job_destruction_incumbents']],
    on=['year', 'dimension', 'category'], how='left'
)

# --- Compute lagged total economy denominators ---
totals = (
    bsd[bsd['dimension'] == 'Total']
    [['year', 'n_firms', 'employment']]
    .rename(columns={'n_firms': 'total_firms', 'employment': 'total_emp'})
    .sort_values('year')
)
totals['lagged_total_firms'] = totals['total_firms'].shift(1)
totals['lagged_total_emp']   = totals['total_emp'].shift(1)

# Merge lagged totals into all rows
bsd = bsd.merge(totals[['year', 'lagged_total_firms', 'lagged_total_emp']], on='year', how='left')

# --- Compute rates (denominator = lagged total economy) ---
bsd['entry_rate']       = (bsd['n_entrants'] + bsd['n_entry_and_exit']) / bsd['lagged_total_firms']
bsd['exit_rate']        = (bsd['n_exiters']  + bsd['n_entry_and_exit']) / bsd['lagged_total_firms']
bsd['jcr_entry']        = bsd['job_creation_entrants']      / bsd['lagged_total_emp']
bsd['jcr_incumbent']    = bsd['job_creation_incumbents']     / bsd['lagged_total_emp']
bsd['jcr_total']        = bsd['jcr_entry'] + bsd['jcr_incumbent']
bsd['jdr_exit']         = bsd['job_destruction_exiters']     / bsd['lagged_total_emp']
bsd['jdr_incumbent']    = bsd['job_destruction_incumbents']  / bsd['lagged_total_emp']
bsd['jdr_total']        = bsd['jdr_exit'] + bsd['jdr_incumbent']
bsd['jrr']              = bsd['jcr_total'] + bsd['jdr_total']

print(f"BSD years: {sorted(bsd['year'].unique())}")
print(f"BSD dimensions: {bsd['dimension'].unique()}")

BSD years: [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
BSD dimensions: ['Age' 'Productivity' 'Region' 'Sector' 'Size' 'Total']


## 2. Load and Prepare ONS Data

In [40]:
JOB_FLOW_COLS = [
    'year', 'group',
    'jcr_total', 'jcr_entry', 'jcr_incumbent',
    'jdr_total', 'jdr_exit', 'jdr_incumbent',
    'net_flow', 'net_entry_exit', 'net_incumbent',
    'jrr_total', 'jrr_entry_exit', 'jrr_incumbent'
]

def load_ons_flow_table(sheet):
    """Load an ONS job-flow table (Tables 6, 10, 13, 19)."""
    df = pd.read_excel(ONS_PATH, sheet_name=sheet, header=None, skiprows=9)
    if df.shape[1] == 13:
        # Table 6 has no group column — insert 'Total'
        df.insert(1, 'group', 'Total')
    df = df.iloc[:, :14]
    df.columns = JOB_FLOW_COLS
    # Drop note/blank rows
    df = df[pd.to_numeric(df['year'], errors='coerce').notna()].copy()
    df['year'] = df['year'].astype(int)
    # Coerce all metric columns (handles '[z]' and other non-numeric markers)
    numeric_cols = JOB_FLOW_COLS[2:]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    return df

# Table 6: total economy job flows
ons_t6  = load_ons_flow_table('Table 6')

# Table 8: entry/exit rates for total economy
ons_t8  = pd.read_excel(ONS_PATH, sheet_name='Table 8', header=None, skiprows=7)
ons_t8  = ons_t8.iloc[:, :7]
ons_t8.columns = ['year', 'n_entrants', 'n_exiters', 'lagged_firms', 'entry_rate', 'exit_rate', 'net_entry']
ons_t8  = ons_t8[pd.to_numeric(ons_t8['year'], errors='coerce').notna()].copy()
ons_t8['year'] = ons_t8['year'].astype(int)
ons_t8[['entry_rate', 'exit_rate']] = ons_t8[['entry_rate', 'exit_rate']].apply(pd.to_numeric, errors='coerce')

# Table 10: by firm size
ons_t10 = load_ons_flow_table('Table 10')

# Table 13: by firm age
ons_t13 = load_ons_flow_table('Table 13')
ons_t13 = ons_t13[ons_t13['group'] != 'Firm age']  # drop header artefact

# Table 19: by sector
ons_t19 = load_ons_flow_table('Table 19')

print("ONS Table 6 shape:",  ons_t6.shape)
print("ONS Table 8 shape:",  ons_t8.shape)
print("ONS Table 10 shape:", ons_t10.shape)
print("ONS Table 13 shape:", ons_t13.shape)
print("ONS Table 19 shape:", ons_t19.shape)
print("\nONS sizes:",  ons_t10['group'].unique())
print("ONS ages:",    ons_t13['group'].unique())
print("\nONS sectors:", ons_t19['group'].unique())

ONS Table 6 shape: (24, 14)
ONS Table 8 shape: (24, 7)
ONS Table 10 shape: (96, 14)
ONS Table 13 shape: (96, 14)
ONS Table 19 shape: (408, 14)

ONS sizes: ['Micro' 'Small' 'Medium' 'Large']
ONS ages: ['New' 'Young' 'Mature' 'Old']

ONS sectors: ['Accommodation & food services' 'Agriculture, forestry & fishing'
 'Arts, entertainment, recreation and other activities'
 'Administration and support services' 'Construction' 'Education'
 'Finance & insurance' 'Human health and social work'
 'Information & communication' 'Other services' 'Manufacturing'
 'Mining, quarrying & utilities' 'Professional, scientific & technical'
 'Real estate activities'
 'Public administration & defence; compulsory social security'
 'Transportation & storage'
 'Wholesale and retail trade; repair of motor vehicles and Motorcycles']


## 3. Sector Mapping (BSD → ONS)

BSD sectors map to ONS industry groups (SIC Section/Division level). Several BSD sectors are sub-components of broader ONS categories, as noted below.

In [41]:
# BSD sector → ONS sector mapping
SECTOR_MAP = {
    'Agriculture':                 'Agriculture, forestry & fishing',
    'Manufacturing':               'Manufacturing',
    'Construction':                'Construction',
    'Wholesale Trade':             'Wholesale and retail trade; repair of motor vehicles and Motorcycles',  # partial (SIC 45-46 only)
    'Retail Trade':                'Wholesale and retail trade; repair of motor vehicles and Motorcycles',  # partial (SIC 47 only)
    'Transport & Logistics':       'Transportation & storage',
    'IT & Computer Services':      'Information & communication',  # partial (SIC 62-63 only)
    'Other Information Services':  'Information & communication',  # partial (SIC 58-61 only)
    'Professional Services':       'Professional, scientific & technical',
    'Business Support Services':   'Administration and support services',
    'Social care':                 'Human health and social work',  # partial (SIC 87-88 only)
    'Utilities':                   'Mining, quarrying & utilities',  # partial (SIC 35-39 only)
    'Hospitality':                 'Accommodation & food services',
    'Other Services':              'Arts, entertainment, recreation and other activities',  # approximate
    'Other Primary Industries':    'Mining, quarrying & utilities',  # partial
}

DIRECT_MAP = {
    'Agriculture':               'Agriculture, forestry & fishing',
    'Manufacturing':             'Manufacturing',
    'Construction':              'Construction',
    'Transport & Logistics':     'Transportation & storage',
    'Professional Services':     'Professional, scientific & technical',
    'Business Support Services': 'Administration and support services',
    'Hospitality':               'Accommodation & food services',
}

bsd_sector = bsd[bsd['dimension'] == 'Sector'].copy()
bsd_sector['ons_sector'] = bsd_sector['category'].map(SECTOR_MAP)

# Aggregate BSD to ONS sector level — sum raw counts across BSD sub-sectors
bsd_sector_agg = (
    bsd_sector
    .groupby(['year', 'ons_sector'], dropna=True)
    .agg(
        n_firms=('n_firms', 'sum'),
        employment=('employment', 'sum'),
        n_entrants=('n_entrants', 'sum'),
        n_exiters=('n_exiters', 'sum'),
        n_entry_and_exit=('n_entry_and_exit', 'sum'),
        jce=('job_creation_entrants', 'sum'),
        jci=('job_creation_incumbents', 'sum'),
        jde=('job_destruction_exiters', 'sum'),
        jdi=('job_destruction_incumbents', 'sum'),
    )
    .reset_index()
    .sort_values(['ons_sector', 'year'])
)

# Compute sector-specific lagged denominators
bsd_sector_agg['lagged_sector_emp']   = bsd_sector_agg.groupby('ons_sector')['employment'].shift(1)
bsd_sector_agg['lagged_sector_firms'] = bsd_sector_agg.groupby('ons_sector')['n_firms'].shift(1)

# Compute rates using sector-specific lagged denominators
bsd_sector_agg['entry_rate']    = (bsd_sector_agg['n_entrants'] + bsd_sector_agg['n_entry_and_exit']) / bsd_sector_agg['lagged_sector_firms']
bsd_sector_agg['exit_rate']     = (bsd_sector_agg['n_exiters']  + bsd_sector_agg['n_entry_and_exit']) / bsd_sector_agg['lagged_sector_firms']
bsd_sector_agg['jcr_entry']     = bsd_sector_agg['jce'] / bsd_sector_agg['lagged_sector_emp']
bsd_sector_agg['jcr_incumbent'] = bsd_sector_agg['jci'] / bsd_sector_agg['lagged_sector_emp']
bsd_sector_agg['jcr_total']     = bsd_sector_agg['jcr_entry'] + bsd_sector_agg['jcr_incumbent']
bsd_sector_agg['jdr_exit']      = bsd_sector_agg['jde'] / bsd_sector_agg['lagged_sector_emp']
bsd_sector_agg['jdr_incumbent'] = bsd_sector_agg['jdi'] / bsd_sector_agg['lagged_sector_emp']
bsd_sector_agg['jdr_total']     = bsd_sector_agg['jdr_exit'] + bsd_sector_agg['jdr_incumbent']
bsd_sector_agg['jrr']           = bsd_sector_agg['jcr_total'] + bsd_sector_agg['jdr_total']

print("BSD sectors mapped to ONS:")
print(bsd_sector_agg['ons_sector'].unique())

BSD sectors mapped to ONS:
['Accommodation & food services' 'Administration and support services'
 'Agriculture, forestry & fishing'
 'Arts, entertainment, recreation and other activities' 'Construction'
 'Human health and social work' 'Information & communication'
 'Manufacturing' 'Mining, quarrying & utilities'
 'Professional, scientific & technical' 'Transportation & storage'
 'Wholesale and retail trade; repair of motor vehicles and Motorcycles']


## 4. Helper Functions

In [42]:
def make_comparison_df(bsd_data, ons_data, bsd_col, ons_col, year_range=(2001, 2022)):
    """Combine BSD and ONS series into long-form for Altair."""
    b = bsd_data[['year', bsd_col]].rename(columns={bsd_col: 'value'})
    b['source'] = 'BSD'
    o = ons_data[['year', ons_col]].rename(columns={ons_col: 'value'})
    o['source'] = 'ONS'
    df = pd.concat([b, o])
    df = df[(df['year'] >= year_range[0]) & (df['year'] <= year_range[1])]
    return df.dropna(subset=['value'])


BSD_COLOUR = pallete['nominal_1']
ONS_COLOUR = pallete['nominal_2']


def line_chart(data, title, y_title='Rate', width=500, height=260):
    """Two-line Altair chart comparing BSD vs ONS."""
    colour_map = alt.Scale(domain=['BSD', 'ONS'], range=[BSD_COLOUR, ONS_COLOUR])
    dash_map   = alt.Scale(domain=['BSD', 'ONS'], range=[[1, 0], [6, 3]])
    return (
        alt.Chart(data)
        .mark_line()
        .encode(
            x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 != 0 ? datum.label : ''",
             labelAngle=0)),
            y=alt.Y('value:Q', title=y_title, axis=alt.Axis(format='.0%')),
            color=alt.Color('source:N', scale=colour_map, legend=alt.Legend(title=None)),
            strokeDash=alt.StrokeDash('source:N', scale=dash_map),
        )
        .properties(title=title, width=width, height=height)
    )


def summary_table(bsd_data, ons_data, bsd_col, ons_col,
                  periods=[(2001,2007,'Pre-GFC'), (2008,2012,'GFC & recovery'),
                            (2013,2019,'Expansion'), (2020,2022,'Covid era')]):
    """Period averages table comparing BSD vs ONS."""
    rows = []
    for start, end, label in periods:
        b_mean = bsd_data[(bsd_data['year']>=start)&(bsd_data['year']<=end)][bsd_col].mean()
        o_mean = ons_data[(ons_data['year']>=start)&(ons_data['year']<=end)][ons_col].mean()
        rows.append({'Period': label, 'BSD': b_mean, 'ONS': o_mean, 'Diff (BSD-ONS)': b_mean - o_mean})
    return pd.DataFrame(rows).set_index('Period').style.format('{:.2%}')

print("Helpers defined.")

Helpers defined.


## 5. Total Economy Comparison

### 5a. Entry and Exit Rates

In [43]:
bsd_total = bsd[(bsd['dimension']=='Total')].sort_values('year')

# Entry rate
df_entry = make_comparison_df(bsd_total, ons_t8, 'entry_rate', 'entry_rate')
chart_entry = line_chart(df_entry, 'Entry Rate — Total Economy')

# Exit rate
df_exit = make_comparison_df(bsd_total, ons_t8, 'exit_rate', 'exit_rate')
chart_exit = line_chart(df_exit, 'Exit Rate — Total Economy')

(chart_entry | chart_exit)

alt.HConcatChart(...)

In [44]:
print("Entry Rate — period averages")
display(summary_table(bsd_total, ons_t8, 'entry_rate', 'entry_rate'))
print("Exit Rate — period averages")
display(summary_table(bsd_total, ons_t8, 'exit_rate', 'exit_rate'))

Entry Rate — period averages


,BSD,ONS,Diff (BSD-ONS)
Period,,,
Pre-GFC,13.70%,13.31%,0.39%
GFC & recovery,11.93%,12.03%,-0.10%
Expansion,14.11%,14.54%,-0.43%
Covid era,12.89%,14.26%,-1.37%


Exit Rate — period averages


,BSD,ONS,Diff (BSD-ONS)
Period,,,
Pre-GFC,13.64%,11.63%,2.00%
GFC & recovery,13.23%,12.61%,0.61%
Expansion,12.87%,11.63%,1.23%
Covid era,13.83%,12.76%,1.07%


### 5b. Job Flow Rates — Total Economy

In [45]:
# JRR
df_jrr = make_comparison_df(bsd_total, ons_t6, 'jrr', 'jrr_total')
c_jrr  = line_chart(df_jrr, 'Job Reallocation Rate')

# JCR total
df_jcr = make_comparison_df(bsd_total, ons_t6, 'jcr_total', 'jcr_total')
c_jcr  = line_chart(df_jcr, 'Job Creation Rate (Total)')

# JDR total
df_jdr = make_comparison_df(bsd_total, ons_t6, 'jdr_total', 'jdr_total')
c_jdr  = line_chart(df_jdr, 'Job Destruction Rate (Total)')

(c_jrr | c_jcr | c_jdr)

alt.HConcatChart(...)

In [46]:
# JCR decomposition: entry vs incumbent
df_jcr_e  = make_comparison_df(bsd_total, ons_t6, 'jcr_entry',     'jcr_entry')
df_jcr_i  = make_comparison_df(bsd_total, ons_t6, 'jcr_incumbent', 'jcr_incumbent')
df_jdr_e  = make_comparison_df(bsd_total, ons_t6, 'jdr_exit',      'jdr_exit')
df_jdr_i  = make_comparison_df(bsd_total, ons_t6, 'jdr_incumbent', 'jdr_incumbent')

c_jcr_e = line_chart(df_jcr_e,  'JCR — Entrants')
c_jcr_i = line_chart(df_jcr_i,  'JCR — Incumbents')
c_jdr_e = line_chart(df_jdr_e,  'JDR — Exiters')
c_jdr_i = line_chart(df_jdr_i,  'JDR — Incumbents')

(c_jcr_e | c_jcr_i) & (c_jdr_e | c_jdr_i)

alt.VConcatChart(...)

In [47]:
metrics = [
    ('jrr',          'jrr_total',       'Job Reallocation Rate'),
    ('jcr_total',    'jcr_total',       'JCR — Total'),
    ('jcr_entry',    'jcr_entry',       'JCR — Entrants'),
    ('jcr_incumbent','jcr_incumbent',   'JCR — Incumbents'),
    ('jdr_total',    'jdr_total',       'JDR — Total'),
    ('jdr_exit',     'jdr_exit',        'JDR — Exiters'),
    ('jdr_incumbent','jdr_incumbent',   'JDR — Incumbents'),
]
for bsd_col, ons_col, label in metrics:
    print(f"\n{label} — period averages")
    display(summary_table(bsd_total, ons_t6, bsd_col, ons_col))


Job Reallocation Rate — period averages


,BSD,ONS,Diff (BSD-ONS)
Period,,,
Pre-GFC,24.93%,25.71%,-0.77%
GFC & recovery,21.01%,21.08%,-0.07%
Expansion,19.98%,20.76%,-0.78%
Covid era,18.03%,19.36%,-1.33%



JCR — Total — period averages


,BSD,ONS,Diff (BSD-ONS)
Period,,,
Pre-GFC,12.70%,13.39%,-0.69%
GFC & recovery,10.47%,10.52%,-0.05%
Expansion,10.98%,11.23%,-0.25%
Covid era,8.87%,9.87%,-1.00%



JCR — Entrants — period averages


...

,BSD,ONS,Diff (BSD-ONS)
Period,,,
Pre-GFC,12.23%,12.31%,-0.08%
GFC & recovery,10.54%,10.55%,-0.02%
Expansion,9.00%,9.53%,-0.53%
Covid era,9.16%,9.49%,-0.33%



JDR — Exiters — period averages


,BSD,ONS,Diff (BSD-ONS)
Period,,,
Pre-GFC,6.34%,4.13%,2.21%
GFC & recovery,4.82%,3.47%,1.35%
Expansion,4.01%,3.06%,0.95%
Covid era,3.56%,3.11%,0.45%



JDR — Incumbents — period averages


,BSD,ONS,Diff (BSD-ONS)
Period,,,
Pre-GFC,5.90%,8.18%,-2.28%
GFC & recovery,5.72%,7.08%,-1.36%
Expansion,4.99%,6.47%,-1.48%
Covid era,5.60%,6.38%,-0.78%


## 6. By Firm Size

Size bands match directly: Micro (0–9), Small (10–49), Medium (50–249), Large (250+).

In [48]:
bsd_size = bsd[bsd['dimension'] == 'Size'].copy()

size_label_map = {
    'Micro (0-9)':     'Micro',
    'Small (10-49)':   'Small',
    'Medium (50-249)': 'Medium',
    'Large (250+)':    'Large',
}
bsd_size['group'] = bsd_size['category'].map(size_label_map)
size_groups = ['Micro', 'Small', 'Medium', 'Large']


def size_facet(bsd_metric, ons_metric, title):
    frames = []
    for g in size_groups:
        b = bsd_size[bsd_size['group'] == g][['year', bsd_metric]].copy()
        b['source'] = 'BSD'; b['group'] = g
        b = b.rename(columns={bsd_metric: 'value'})

        o = ons_t10[ons_t10['group'] == g][['year', ons_metric]].copy()
        o['source'] = 'ONS'; o['group'] = g
        o = o.rename(columns={ons_metric: 'value'})
        frames.extend([b, o])

    df = pd.concat(frames)
    df = df[(df['year'] >= 2001) & (df['year'] <= 2022)].dropna(subset=['value'])

    colour_map = alt.Scale(domain=['BSD','ONS'], range=[BSD_COLOUR, ONS_COLOUR])
    dash_map   = alt.Scale(domain=['BSD','ONS'], range=[[1,0],[6,3]])

    return (
        alt.Chart(df)
        .mark_line()
        .encode(
            x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 != 0 ? datum.label : ''",
             labelAngle=0), title=None),
            y=alt.Y('value:Q', title='Rate', axis=alt.Axis(format='.0%')),
            color=alt.Color('source:N', scale=colour_map, legend=alt.Legend(title=None)),
            strokeDash=alt.StrokeDash('source:N', scale=dash_map),
            facet=alt.Facet('group:N', title=None, sort=size_groups, column=2),
        )
        .properties(title=title)
        .resolve_scale(y='independent')
    )

size_facet('jcr_total', 'jcr_total', 'Job Creation Rate by Firm Size')

SchemaValidationError: `Facet` has no parameter named 'column'

Existing parameter names are:
shorthand      bin       field     timeUnit   
aggregate      bounds    header    title      
align          center    sort      type       
bandPosition   columns   spacing              

See the help for `Facet` to read the full description of these parameters

alt.Chart(...)

In [49]:
size_facet('jdr_total', 'jdr_total', 'Job Destruction Rate by Firm Size')

SchemaValidationError: `Facet` has no parameter named 'column'

Existing parameter names are:
shorthand      bin       field     timeUnit   
aggregate      bounds    header    title      
align          center    sort      type       
bandPosition   columns   spacing              

See the help for `Facet` to read the full description of these parameters

alt.Chart(...)

In [50]:
size_facet('jrr', 'jrr_total', 'Job Reallocation Rate by Firm Size')

SchemaValidationError: `Facet` has no parameter named 'column'

Existing parameter names are:
shorthand      bin       field     timeUnit   
aggregate      bounds    header    title      
align          center    sort      type       
bandPosition   columns   spacing              

See the help for `Facet` to read the full description of these parameters

alt.Chart(...)

In [51]:
size_facet('jcr_entry', 'jcr_entry', 'JCR — Entry Contribution by Firm Size')

SchemaValidationError: `Facet` has no parameter named 'column'

Existing parameter names are:
shorthand      bin       field     timeUnit   
aggregate      bounds    header    title      
align          center    sort      type       
bandPosition   columns   spacing              

See the help for `Facet` to read the full description of these parameters

alt.Chart(...)

In [52]:
size_facet('jcr_incumbent', 'jcr_incumbent', 'JCR — Incumbent Contribution by Firm Size')

SchemaValidationError: `Facet` has no parameter named 'column'

Existing parameter names are:
shorthand      bin       field     timeUnit   
aggregate      bounds    header    title      
align          center    sort      type       
bandPosition   columns   spacing              

See the help for `Facet` to read the full description of these parameters

alt.Chart(...)

In [53]:
size_facet('jdr_exit', 'jdr_exit', 'JDR — Exit Contribution by Firm Size')

SchemaValidationError: `Facet` has no parameter named 'column'

Existing parameter names are:
shorthand      bin       field     timeUnit   
aggregate      bounds    header    title      
align          center    sort      type       
bandPosition   columns   spacing              

See the help for `Facet` to read the full description of these parameters

alt.Chart(...)

In [54]:
size_facet('jdr_incumbent', 'jdr_incumbent', 'JDR — Incumbent Contribution by Firm Size')

SchemaValidationError: `Facet` has no parameter named 'column'

Existing parameter names are:
shorthand      bin       field     timeUnit   
aggregate      bounds    header    title      
align          center    sort      type       
bandPosition   columns   spacing              

See the help for `Facet` to read the full description of these parameters

alt.Chart(...)

In [55]:
# Summary tables by size
size_metrics = [
    ('jrr',          'jrr_total',    'JRR'),
    ('jcr_total',    'jcr_total',    'JCR Total'),
    ('jcr_entry',    'jcr_entry',    'JCR Entry'),
    ('jcr_incumbent','jcr_incumbent','JCR Incumbent'),
    ('jdr_total',    'jdr_total',    'JDR Total'),
    ('jdr_exit',     'jdr_exit',     'JDR Exit'),
    ('jdr_incumbent','jdr_incumbent','JDR Incumbent'),
]
for g in size_groups:
    b_g = bsd_size[bsd_size['group'] == g]
    o_g = ons_t10[ons_t10['group'] == g]
    rows = []
    for bsd_m, ons_m, label in size_metrics:
        b_mean = b_g[(b_g['year']>=2001)&(b_g['year']<=2022)][bsd_m].mean()
        o_mean = o_g[(o_g['year']>=2001)&(o_g['year']<=2022)][ons_m].mean()
        rows.append({'Measure': label, 'BSD': b_mean, 'ONS': o_mean, 'Diff': b_mean - o_mean})
    tbl = pd.DataFrame(rows).set_index('Measure')
    print(f"\n=== {g} (2001-2022 average) ===")
    display(tbl.style.format('{:.2%}'))


=== Micro (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,7.08%,8.17%,-1.08%
JCR Total,3.23%,3.19%,0.04%
JCR Entry,1.87%,2.03%,-0.17%
JCR Incumbent,1.36%,1.16%,0.20%
JDR Total,3.86%,4.98%,-1.12%
JDR Exit,2.02%,1.81%,0.21%
JDR Incumbent,1.83%,3.16%,-1.33%



=== Small (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,3.83%,3.37%,0.46%
JCR Total,2.10%,1.93%,0.18%
JCR Entry,0.61%,0.64%,-0.02%
JCR Incumbent,1.49%,1.29%,0.20%
JDR Total,1.73%,1.44%,0.28%
JDR Exit,0.91%,0.66%,0.25%
JDR Incumbent,0.82%,0.78%,0.04%



=== Medium (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,3.20%,2.80%,0.40%
JCR Total,1.71%,1.68%,0.03%
JCR Entry,0.37%,0.47%,-0.10%
JCR Incumbent,1.34%,1.22%,0.13%
JDR Total,1.49%,1.12%,0.37%
JDR Exit,0.69%,0.38%,0.31%
JDR Incumbent,0.80%,0.74%,0.06%



=== Large (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,7.41%,7.88%,-0.46%
JCR Total,4.08%,4.77%,-0.68%
JCR Entry,0.65%,0.84%,-0.19%
JCR Incumbent,3.43%,3.93%,-0.50%
JDR Total,3.33%,3.11%,0.22%
JDR Exit,1.26%,0.66%,0.60%
JDR Incumbent,2.07%,2.45%,-0.38%


## 7. By Firm Age

Age bands match: New (0–2 yrs), Young (3–5 yrs), Mature (6–10 yrs), Old (11+ yrs).

**Note:** ONS marks entering-business contributions as `[z]` (not applicable) for Young, Mature and Old age groups — only New firms contribute to entry-side job creation. BSD reflects the same logic since entrant counts for non-new cohorts are zero.

In [56]:
bsd_age = bsd[bsd['dimension'] == 'Age'].copy()

age_label_map = {
    'New (0-2 years)':    'New',
    'Young (3-5 years)':  'Young',
    'Mature (6-10 years)':'Mature',
    'Old (11+ years)':    'Old',
}
bsd_age['group'] = bsd_age['category'].map(age_label_map)
age_groups = ['New', 'Young', 'Mature', 'Old']


def age_facet(bsd_metric, ons_metric, title):
    frames = []
    for g in age_groups:
        b = bsd_age[bsd_age['group'] == g][['year', bsd_metric]].copy()
        b['source'] = 'BSD'; b['group'] = g
        b = b.rename(columns={bsd_metric: 'value'})

        o = ons_t13[ons_t13['group'] == g][['year', ons_metric]].copy()
        o['source'] = 'ONS'; o['group'] = g
        o = o.rename(columns={ons_metric: 'value'})
        frames.extend([b, o])

    df = pd.concat(frames)
    df = df[(df['year'] >= 2001) & (df['year'] <= 2022)].dropna(subset=['value'])

    colour_map = alt.Scale(domain=['BSD','ONS'], range=[BSD_COLOUR, ONS_COLOUR])
    dash_map   = alt.Scale(domain=['BSD','ONS'], range=[[1,0],[6,3]])

    return (
        alt.Chart(df)
        .mark_line()
        .encode(
            x=alt.X('year:O', , axis=alt.Axis(labelExpr="datum.value % 2 != 0 ? datum.label : ''",
             labelAngle=0), title=None),
            y=alt.Y('value:Q', title='Rate', axis=alt.Axis(format='.0%')),
            color=alt.Color('source:N', scale=colour_map, legend=alt.Legend(title=None)),
            strokeDash=alt.StrokeDash('source:N', scale=dash_map),
            facet=alt.Facet('group:N', title=None, sort=age_groups, columns=2),
        )
        .properties(title=title)
        .resolve_scale(y='independent')
    )

age_facet('jcr_total', 'jcr_total', 'Job Creation Rate by Firm Age')

SyntaxError: invalid syntax (3151360846.py, line 35)

In [ ]:
age_facet('jdr_total', 'jdr_total', 'Job Destruction Rate by Firm Age')

alt.Chart(...)

In [ ]:
age_facet('jrr', 'jrr_total', 'Job Reallocation Rate by Firm Age')

alt.Chart(...)

In [ ]:
age_facet('jcr_entry', 'jcr_entry', 'JCR — Entry Contribution by Firm Age')

alt.Chart(...)

In [ ]:
age_facet('jcr_incumbent', 'jcr_incumbent', 'JCR — Incumbent Contribution by Firm Age')

alt.Chart(...)

In [ ]:
age_facet('jdr_exit', 'jdr_exit', 'JDR — Exit Contribution by Firm Age')

alt.Chart(...)

In [ ]:
age_facet('jdr_incumbent', 'jdr_incumbent', 'JDR — Incumbent Contribution by Firm Age')

alt.Chart(...)

In [ ]:
# Summary tables by age
for g in age_groups:
    b_g = bsd_age[bsd_age['group'] == g]
    o_g = ons_t13[ons_t13['group'] == g]
    rows = []
    for bsd_m, ons_m, label in size_metrics:
        b_mean = b_g[(b_g['year']>=2001)&(b_g['year']<=2022)][bsd_m].mean()
        o_mean = o_g[(o_g['year']>=2001)&(o_g['year']<=2022)][ons_m].mean()
        rows.append({'Measure': label, 'BSD': b_mean, 'ONS': o_mean, 'Diff': b_mean - o_mean})
    tbl = pd.DataFrame(rows).set_index('Measure')
    print(f"\n=== {g} firms (2001-2022 average) ===")
    display(tbl.style.format('{:.2%}'))


=== New firms (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,6.50%,7.52%,-1.02%
JCR Total,4.65%,5.16%,-0.51%
JCR Entry,3.50%,3.98%,-0.48%
JCR Incumbent,1.16%,1.18%,-0.03%
JDR Total,1.85%,2.36%,-0.51%
JDR Exit,1.09%,1.24%,-0.15%
JDR Incumbent,0.76%,1.12%,-0.36%



=== Young firms (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,2.63%,2.56%,0.06%
JCR Total,1.13%,1.09%,0.04%
JCR Entry,0.00%,nan%,nan%
JCR Incumbent,1.13%,1.09%,0.04%
JDR Total,1.50%,1.47%,0.03%
JDR Exit,0.88%,0.67%,0.21%
JDR Incumbent,0.62%,0.81%,-0.18%



=== Mature firms (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,2.34%,2.91%,-0.57%
JCR Total,1.01%,1.29%,-0.28%
JCR Entry,0.00%,nan%,nan%
JCR Incumbent,1.01%,1.29%,-0.28%
JDR Total,1.34%,1.62%,-0.28%
JDR Exit,0.70%,0.57%,0.13%
JDR Incumbent,0.64%,1.05%,-0.41%



=== Old firms (2001-2022 average) ===


,BSD,ONS,Diff
Measure,,,
JRR,10.05%,9.22%,0.83%
JCR Total,4.33%,4.03%,0.31%
JCR Entry,0.00%,nan%,nan%
JCR Incumbent,4.33%,4.03%,0.31%
JDR Total,5.72%,5.19%,0.52%
JDR Exit,2.21%,1.02%,1.18%
JDR Incumbent,3.51%,4.17%,-0.66%


## 8. By Sector

Sector comparisons are **approximate** — BSD uses a custom sector grouping (SIC division-level) mapped to the ONS SIC section-level industry groups. Where BSD sectors are sub-components of ONS groups (e.g. Wholesale + Retail → ONS 'Wholesale and retail trade'), BSD numerators are summed before computing rates.

Sectors with **direct (one-to-one) mapping**: Agriculture, Manufacturing, Construction, Transport & Logistics, Professional Services, Business Support Services, Hospitality.

Sectors with **partial/approximate mapping**: Wholesale+Retail Trade → ONS Wholesale & retail, IT+Other Information → ONS Information & communication, Utilities+Other Primary → ONS Mining, quarrying & utilities, Social care → ONS Human health & social work.

In [ ]:
ons_sector_groups = sorted(bsd_sector_agg['ons_sector'].dropna().unique())


def sector_comparison(bsd_metric, ons_metric, title, sectors=None):
    if sectors is None:
        sectors = ons_sector_groups

    frames = []
    for s in sectors:
        short = s[:35]
        b = bsd_sector_agg[bsd_sector_agg['ons_sector'] == s][['year', bsd_metric]].copy()
        b['source'] = 'BSD'; b['group'] = short
        b = b.rename(columns={bsd_metric: 'value'})

        o = ons_t19[ons_t19['group'] == s][['year', ons_metric]].copy()
        o['source'] = 'ONS'; o['group'] = short
        o = o.rename(columns={ons_metric: 'value'})
        frames.extend([b, o])

    df = pd.concat(frames)
    df = df[(df['year'] >= 2001) & (df['year'] <= 2022)].dropna(subset=['value'])

    colour_map = alt.Scale(domain=['BSD','ONS'], range=[BSD_COLOUR, ONS_COLOUR])
    dash_map   = alt.Scale(domain=['BSD','ONS'], range=[[1,0],[6,3]])

    return (
        alt.Chart(df)
        .mark_line()
        .encode(
            x=alt.X('year:O', , axis=alt.Axis(labelExpr="datum.value % 2 != 0 ? datum.label : ''",
             labelAngle=0), title=None),
            y=alt.Y('value:Q', title='Rate', axis=alt.Axis(format='.0%')),
            color=alt.Color('source:N', scale=colour_map, legend=alt.Legend(title=None)),
            strokeDash=alt.StrokeDash('source:N', scale=dash_map),
            facet=alt.Facet('group:N', title=None, columns=3),
        )
        .properties(title=title, width=260, height=160)
        .resolve_scale(y='independent')
    )

sector_comparison('jrr', 'jrr_total', 'Job Reallocation Rate by Sector')

alt.Chart(...)

In [ ]:
sector_comparison('jcr_total', 'jcr_total', 'Job Creation Rate by Sector')

alt.Chart(...)

In [ ]:
sector_comparison('jdr_total', 'jdr_total', 'Job Destruction Rate by Sector')

alt.Chart(...)

In [ ]:
sector_comparison('jcr_entry', 'jcr_entry', 'JCR — Entry Contribution by Sector')

alt.Chart(...)

In [ ]:
sector_comparison('jcr_incumbent', 'jcr_incumbent', 'JCR — Incumbent Contribution by Sector')

alt.Chart(...)

In [ ]:
sector_comparison('jdr_exit', 'jdr_exit', 'JDR — Exit Contribution by Sector')

alt.Chart(...)

In [ ]:
sector_comparison('jdr_incumbent', 'jdr_incumbent', 'JDR — Incumbent Contribution by Sector')

alt.Chart(...)

In [ ]:
# Summary table: sector-level 2001-2022 averages
sector_metrics = [
    ('jrr',          'jrr_total',    'JRR'),
    ('jcr_total',    'jcr_total',    'JCR Total'),
    ('jcr_entry',    'jcr_entry',    'JCR Entry'),
    ('jcr_incumbent','jcr_incumbent','JCR Incumbent'),
    ('jdr_total',    'jdr_total',    'JDR Total'),
    ('jdr_exit',     'jdr_exit',     'JDR Exit'),
    ('jdr_incumbent','jdr_incumbent','JDR Incumbent'),
]
rows = []
for s in ons_sector_groups:
    short = s[:40]
    b_g = bsd_sector_agg[(bsd_sector_agg['ons_sector']==s)&(bsd_sector_agg['year']>=2001)&(bsd_sector_agg['year']<=2022)]
    o_g = ons_t19[(ons_t19['group']==s)&(ons_t19['year']>=2001)&(ons_t19['year']<=2022)]
    for bsd_m, ons_m, label in sector_metrics:
        b_mean = b_g[bsd_m].mean()
        o_mean = o_g[ons_m].mean()
        rows.append({
            'Sector': short,
            'Measure': label,
            'BSD': b_mean,
            'ONS': o_mean,
            'Diff (BSD-ONS)': b_mean - o_mean
        })

sector_tbl = pd.DataFrame(rows).pivot_table(
    index='Sector', columns='Measure', values='Diff (BSD-ONS)'
)[['JRR','JCR Total','JCR Entry','JCR Incumbent','JDR Total','JDR Exit','JDR Incumbent']]

print("Sector-level differences (BSD minus ONS), 2001-2022 averages")
display(sector_tbl.style.format('{:.2%}').background_gradient(cmap='RdYlGn', axis=None, vmin=-0.03, vmax=0.03))

Sector-level differences (BSD minus ONS), 2001-2022 averages


Measure,JRR,JCR Total,JCR Entry,JCR Incumbent,JDR Total,JDR Exit,JDR Incumbent
Sector,,,,,,,
Accommodation & food services,-3.46%,-1.75%,-1.43%,-0.33%,-1.71%,0.44%,-2.15%
Administration and support services,-4.68%,-3.00%,-2.09%,-0.92%,-1.67%,1.21%,-2.88%
"Agriculture, forestry & fishing",-1.92%,-1.39%,-1.09%,-0.30%,-0.53%,0.05%,-0.58%
"Arts, entertainment, recreation and othe",-8.28%,-4.53%,-3.90%,-0.62%,-3.75%,-1.93%,-1.82%
Construction,-3.66%,-2.03%,-1.64%,-0.39%,-1.63%,0.42%,-2.04%
Human health and social work,3.41%,1.20%,1.01%,0.18%,2.21%,3.87%,-1.66%
Information & communication,-3.28%,-1.49%,-0.90%,-0.59%,-1.79%,0.64%,-2.43%
Manufacturing,-2.47%,-1.10%,-0.58%,-0.52%,-1.37%,0.35%,-1.72%
"Mining, quarrying & utilities",-3.59%,-2.00%,-1.35%,-0.64%,-1.60%,1.30%,-2.89%


## 9. Entry/Exit Rates for Direct-Mapped Sectors

Entry and exit rate breakdowns are not published by ONS in the sector/size/age tables — the ONS only publishes firm-count-based entry/exit rates for the total economy (Table 8). The BSD sector-level entry and exit rates are shown below for reference only.

In [ ]:
bsd_sector_direct = bsd_sector[bsd_sector['category'].isin(DIRECT_MAP.keys())].copy()
bsd_sector_direct = bsd_sector_direct[(bsd_sector_direct['year']>=2001)&(bsd_sector_direct['year']<=2022)]

sector_colour_range = [
    pallete['nominal_1'], pallete['nominal_2'], pallete['nominal_3'],
    pallete['nominal_4'], pallete['nominal_5'], pallete['nominal_6'],
    pallete.get('accent', '#888888'),
]
colour_map = alt.Scale(
    domain=list(DIRECT_MAP.keys()),
    range=sector_colour_range[:len(DIRECT_MAP)]
)

def sector_entry_exit_chart(metric, title):
    return (
        alt.Chart(bsd_sector_direct)
        .mark_line(point=True, size=2)
        .encode(
            x=alt.X('year:O', title='Year'),
            y=alt.Y(f'{metric}:Q', title='Rate', axis=alt.Axis(format='.0%')),
            color=alt.Color('category:N', scale=colour_map, legend=alt.Legend(title='Sector')),
        )
        .properties(title=title, width=600, height=280)
    )

sector_entry_exit_chart('entry_rate', 'BSD Entry Rate by Sector (direct-mapped sectors)')

alt.Chart(...)

In [ ]:
sector_entry_exit_chart('exit_rate', 'BSD Exit Rate by Sector (direct-mapped sectors)')

alt.Chart(...)

## 10. Validation Summary

Correlation and mean absolute difference across all years for the total-economy series.

In [ ]:
total_metrics = [
    ('entry_rate', 'entry_rate',    'Entry rate',       bsd_total, ons_t8),
    ('exit_rate',  'exit_rate',     'Exit rate',        bsd_total, ons_t8),
    ('jrr',        'jrr_total',     'JRR',              bsd_total, ons_t6),
    ('jcr_total',  'jcr_total',     'JCR Total',        bsd_total, ons_t6),
    ('jcr_entry',  'jcr_entry',     'JCR Entry',        bsd_total, ons_t6),
    ('jcr_incumbent','jcr_incumbent','JCR Incumbent',   bsd_total, ons_t6),
    ('jdr_total',  'jdr_total',     'JDR Total',        bsd_total, ons_t6),
    ('jdr_exit',   'jdr_exit',      'JDR Exit',         bsd_total, ons_t6),
    ('jdr_incumbent','jdr_incumbent','JDR Incumbent',   bsd_total, ons_t6),
]

summary_rows = []
for bsd_m, ons_m, label, bsd_df, ons_df in total_metrics:
    merged = (
        bsd_df[['year', bsd_m]].rename(columns={bsd_m: 'bsd_val'})
        .merge(ons_df[['year', ons_m]].rename(columns={ons_m: 'ons_val'}), on='year')
        .query('year >= 2001 and year <= 2022')
        .dropna()
    )
    if len(merged) < 3:
        continue
    corr   = merged['bsd_val'].corr(merged['ons_val'])
    mad    = (merged['bsd_val'] - merged['ons_val']).abs().mean()
    bias   = (merged['bsd_val'] - merged['ons_val']).mean()
    b_mean = merged['bsd_val'].mean()
    o_mean = merged['ons_val'].mean()
    summary_rows.append({
        'Measure':             label,
        'BSD mean':            b_mean,
        'ONS mean':            o_mean,
        'Mean bias (BSD-ONS)': bias,
        'Mean abs. diff.':     mad,
        'Correlation':         corr,
    })

summary_df = pd.DataFrame(summary_rows).set_index('Measure')
display(
    summary_df.style
    .format({'BSD mean': '{:.2%}', 'ONS mean': '{:.2%}',
             'Mean bias (BSD-ONS)': '{:.2%}', 'Mean abs. diff.': '{:.2%}',
             'Correlation': '{:.3f}'})
    .background_gradient(subset=['Correlation'], cmap='Blues', vmin=0.5, vmax=1)
    .background_gradient(subset=['Mean abs. diff.'], cmap='YlOrRd_r', vmin=0, vmax=0.02)
)

,BSD mean,ONS mean,Mean bias (BSD-ONS),Mean abs. diff.,Correlation
Measure,,,,,
Entry rate,13.32%,13.54%,-0.22%,1.60%,0.326
Exit rate,13.33%,12.01%,1.32%,1.64%,-0.179
JRR,21.52%,22.21%,-0.69%,1.57%,0.786
JCR Total,11.12%,11.57%,-0.45%,1.00%,0.696
JCR Entry,3.50%,3.98%,-0.48%,0.62%,0.540
JCR Incumbent,7.63%,7.60%,0.03%,0.80%,0.641
JDR Total,10.40%,10.64%,-0.24%,0.74%,0.792
JDR Exit,4.87%,3.50%,1.37%,1.39%,0.611
JDR Incumbent,5.53%,7.14%,-1.61%,1.61%,0.503
